In [ ]:
%pip install -q python-dotenv openai openpyxl langchain-community langchain-text-splitters sentence-transformers faiss-cpu pypdf cryptography

Note: you may need to restart the kernel to use updated packages.


In [44]:
# imports
import os
import time
from typing import Dict, List

import pandas as pd
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI


In [ ]:
# constants

COLAB = True

if COLAB:
    from google.colab import userdata

    NEBIUS_API_KEY = userdata.get("NEBIUS_API_KEY")
else:
    load_dotenv()
    NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")

NEBIUS_BASE_URL = "https://api.studio.nebius.ai/v1/"
PDFS_PATH = "https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs"
RAG_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
JUDGE_MODEL = ""
VECTORSTORE_DIR = "financebench_rag_faiss"

client = OpenAI(api_key=NEBIUS_API_KEY, base_url=NEBIUS_BASE_URL)
print("Client ready ✓")

Client ready ✓


In [29]:
# dataset
def get_dataset():
    df = pd.read_json(
        "hf://datasets/PatronusAI/financebench/financebench_merged.jsonl",
        lines=True,
    )
    # Drop the metrics-generated questions
    df = df[df["question_type"] != "metrics-generated"]

    # For each row, replace the "doc_link" value with the url from "https://github.com/patronus-ai/financebench/tree/main/pdfs" such that the url is: "https://github.com/patronus-ai/financebench/tree/main/pdfs/{{doc_name}}.pdf"
    df["doc_link"] = df["doc_name"].apply(lambda x: f"{PDFS_PATH}/{x}.pdf")
    df = df.sort_values(by="financebench_id", ascending=True).reset_index(drop=True)

    return df


df = get_dataset()
print("Columns:", df.columns.tolist())
print("Unique question types:", df.question_type.unique().tolist())
df.head(2)

Columns: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link']
Unique question types: ['domain-relevant', 'novel-generated']


,financebench_id,company,doc_name,question_type,question_reasoning,domain_question_num,question,answer,justification,dataset_subset_label,evidence,gics_sector,doc_type,doc_period,doc_link
0,financebench_id_00005,Corning,CORNING_2022_10K,domain-relevant,Numerical reasoning OR Logical reasoning,dg24,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,"Trade accounts receivable, net of doubtful acc...",OPEN_SOURCE,[{'evidence_text': 'Consolidated Balance Sheet...,Information Technology,10k,2022,https://raw.githubusercontent.com/patronus-ai/...
1,financebench_id_00070,American Water Works,AMERICANWATERWORKS_2022_10K,domain-relevant,Numerical reasoning OR Logical reasoning,dg24,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",Accounts receivable+Income tax receivable+Unbi...,OPEN_SOURCE,[{'evidence_text': 'American Water Works Compa...,Utilities,10k,2022,https://raw.githubusercontent.com/patronus-ai/...


---
## Task 1 - Naive Generation

In [31]:
# answer the first 5 questions of each question_type - 5 domain-relevant, 5 novel-generated

TASK_1_FILENAME = "assignment2_naive_generation.xlsx"

if os.path.exists(TASK_1_FILENAME):
    print("Loading existing results...")
    answers_df = pd.read_excel(TASK_1_FILENAME)
else:
    # Select the first 5 questions for each question_type
    questions_domain = df[df["question_type"] == "domain-relevant"].head(5)
    questions_novel = df[df["question_type"] == "novel-generated"].head(5)
    selected_questions = pd.concat([questions_domain, questions_novel]).reset_index(
        drop=True
    )

    answers = []

    for idx, row in selected_questions.iterrows():
        prompt = f"""
Answer the question in 2-4 sentences.
If you don't know the answer, say "I don't know".
QUESTION: {row["question"]}
ANSWER:
"""
        response = client.chat.completions.create(
            model=RAG_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=200,
        )
        naive_answer = response.choices[0].message.content.strip()
        result = pd.Series(
            {
                "financebench_id": row["financebench_id"],
                "question_type": row["question_type"],
                "question": row["question"],
                "naive_answer": naive_answer,
                "ground_truth": row["answer"],
                "verdict": "",  # correct/partially correct/wrong/refused
            }
        )
        answers.append(result)
        time.sleep(1.5)  # rate limiting

    answers_df = pd.DataFrame(answers)

    # save results before setting the verdict
    answers_df.to_excel(TASK_1_FILENAME, index=False)

Loading existing results...


In [6]:
# set verdict
VERDICT_MAP = {
    1: "correct",
    2: "partially correct",
    3: "wrong",
    4: "refused",
}
verdicts_dict = {
    "financebench_id_00005": VERDICT_MAP[1],
    "financebench_id_00070": VERDICT_MAP[4],
    "financebench_id_00080": VERDICT_MAP[1],
    "financebench_id_00206": VERDICT_MAP[1],
    "financebench_id_00215": VERDICT_MAP[2],
    "financebench_id_00283": VERDICT_MAP[3],
    "financebench_id_00288": VERDICT_MAP[4],
    "financebench_id_00299": VERDICT_MAP[4],
    "financebench_id_00302": VERDICT_MAP[4],
    "financebench_id_00382": VERDICT_MAP[2],
}

for fid, verdict in verdicts_dict.items():
    answers_df.loc[answers_df["financebench_id"] == fid, "verdict"] = verdict

answers_df.to_excel(TASK_1_FILENAME, index=False)

In [32]:
answers_df.head(2)

,financebench_id,question_type,question,naive_answer,ground_truth,verdict
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,"Based on Corning's FY2022 data, the company ha...",Yes. Corning had a positive working capital am...,correct
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,I don't know the specific details of American ...,"No, American Water Works had negative working ...",refused


#### Questions:

1. Cases where the model **refused** or asked for more information - why?
- In `novel-generated` Q.s, the model refused to answer (not enough knowledge) 3 times as opposed to only once for the given `domain-relevant` Q.s.<br>
I don't see a reason why sometimes it refuses to answer while sometime it hallucinates some answer...<br>
However, it DOES refuse because we told it in the prompt ('If you don't know the answer, say "I don't know"') - and it really does not have any relevant information for any of these questions.

2. Cases where the model **answered confidently** - spot-check against the ground truth. Is the answer correct? Partially correct? Totally wrong (hallucinated)?
- Overall, the model is confident with its answers, regardless of correctness. Even when it refuses to answer - it explains why it cannot answer. 
The one time it was completely wrong was when we asked it for a specific number ("How much ... **in USD million?**") - it just output some random number and was therefore wrong ("$12 billion"). For other questions, it was more like 50-50% ("Does Corning have positive working capital?") - and its best chance to succeed is to just say "yes"/"no". But it's worth as guessing.<br>
This is not surprising because that is the behaviour I encounter since starting using LLMs a couple of years ago - they always answer regardless of the data they have, mostly with a concrete answer, even when completely wrong. This holds also for the best models today.

3. Are there patterns by *question_type*? Do some types fail more than others?
- The `domain-relevant` Q.s are more yes/no Q.s.<br>
Therefore, with the naive answers we received - I see separation for the model's answers by the questions types - it answered this type of questions (but not necessarily correct).<br>
Regarding the `novel-generated` questions, those that are open questions (asking for an amount/specific segments names), it often refused to answer with "I don't know" - probably becasue it can't guess yes/no.<br>

---
## Task 2 - RAG Reminder

### RAG Pipeline Components

#### Indexing (Documents → Chunk + Embed → Vector Store)
**Contribution:** Transforms the raw corpus into a searchable knowledge base by splitting documents into manageable chunks and mapping each chunk to a dense vector in embedding space, so that semantic similarity search becomes possible later. This builds the vector store $D$ that "retrieval" will query against.

**Failure modes:** Poor chunking (e.g., splitting mid-sentence or using chunks too large/small) destroys semantic coherence - a chunk that spans two unrelated topics produces a messy embedding. The embedding model may be domain-mismatched (e.g., a general-purpose encoder on legal or medical text), causing near-duplicate documents to land far apart. Other issues: stale/missing documents, lost metadata (page numbers, titles), OCR errors, or inconsistent preprocessing between indexing time and query time.

**When:** Happens **once, offline** (with periodic re-indexing when the corpus changes). This is the most expensive step per document but amortized across all future queries.


#### Retrieval ($\Gamma$: User Query → Top-k Chunks)
**Contribution:** Given a query $q$, embeds it with the *same* encoder used at indexing and searches the vector store $D$ to return the top-$k$ most relevant chunks. This grounds the generator in specific, query-relevant evidence rather than relying purely on parametric memory.

**Failure modes:** Query-document vocabulary mismatch (user asks "how do I cancel?" but docs say "terminate subscription") - pure dense retrieval can miss this, which is why hybrid BM25+dense often helps. Wrong $k$: too small misses key context, too large dilutes the prompt with noise and wastes tokens. Other issues: multi-hop questions that need information spread across chunks, ambiguous queries, or an embedding mismatch between query-time and index-time encoders.

**When:** **Per query** - runs on every user request. Latency-sensitive, so ANN indices (FAISS, HNSW) are typically used instead of exact search.


#### Generation ($\Theta$: Query + Retrieved Chunks → Answer)
**Contribution:** An LLM consumes the query plus retrieved context and synthesizes a grounded natural-language answer, ideally citing or quoting the retrieved evidence. This is where retrieved facts become a user-facing response.

**Failure modes:** **Hallucination** even with correct context (the model ignores retrieved chunks and invents facts), or the opposite - the retrieved context *is* wrong/irrelevant and the model faithfully parrots it ("garbage in, garbage out"). Prompt-budget issues: context gets truncated and the crucial chunk is dropped. Also: "lost in the middle" (LLMs under-attend to mid-context chunks, as Yuval stated in class), stale context not reflecting the latest query intent, or tone/format drift from the system prompt.

**When:** **Per query** - one (or more) LLM calls per user request. Usually the dominant cost/latency component of the pipeline.

---
## Task 3 - Embed Documents

In [40]:
# Task 3
doc_rows = df.drop_duplicates(subset=["doc_name"], keep="first")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

if os.path.isdir(VECTORSTORE_DIR):
    vectorstore = FAISS.load_local(
        VECTORSTORE_DIR,
        embeddings,
        allow_dangerous_deserialization=True,
    )
    print(
        f"Loaded FAISS index from {VECTORSTORE_DIR!r} ({vectorstore.index.ntotal} vectors)"
    )
else:
    # Load only PDFs for doc_name values in the dataset; metadata on each page before split
    all_pages = []

    for _, row in doc_rows.iterrows():
        loader = PyPDFLoader(row["doc_link"])
        pages = loader.load()

        for page_number, doc in enumerate(pages):
            doc.metadata = doc.metadata or {}
            doc.metadata["doc_name"] = row["doc_name"]
            doc.metadata["company"] = row["company"]
            doc.metadata["doc_period"] = row["doc_period"]
            doc.metadata["page_number"] = (
                page_number  # 0-indexed to match the dataset's evidence_page_num
            )
            all_pages.append(doc)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
    )
    chunks = text_splitter.split_documents(all_pages)
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(VECTORSTORE_DIR)
    print(
        f"Built and saved FAISS index to {VECTORSTORE_DIR!r} ({vectorstore.index.ntotal} vectors)"
    )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded FAISS index from 'financebench_rag_faiss' (24218 vectors)


In [42]:
def check_retrieval(
    row: pd.Series, vectorstore, top_k: int = 5, verbose: bool = True
) -> dict:
    """Retrieve top-k chunks for a dataset row and check doc, page, and evidence overlap."""
    query = row["question"]
    expected_doc = row["doc_name"]

    evidence_texts, evidence_pages = [], set()
    for item in row.get("evidence", []):
        if isinstance(item, dict):
            txt = item.get("evidence_text", "")
            if txt:
                evidence_texts.append(txt)
            pg = item.get("evidence_page_num")
            if pg is not None:
                evidence_pages.add(int(pg))

    retrieved = vectorstore.similarity_search(query, k=top_k)

    doc_hits = [d for d in retrieved if d.metadata.get("doc_name") == expected_doc]
    page_hits = [
        d for d in retrieved if d.metadata.get("page_number") in evidence_pages
    ]
    overlap_hits = [
        d
        for d in retrieved
        if any(
            ev.lower()[:180] in d.page_content.lower()
            or d.page_content.lower()[:180] in ev.lower()
            for ev in evidence_texts
            if ev
        )
    ]

    result = {
        "question": query,
        "expected_doc": expected_doc,
        "evidence_pages": sorted(evidence_pages),
        "doc_match": len(doc_hits),
        "page_match": len(page_hits),
        "evidence_overlap": len(overlap_hits),
        "top_k": top_k,
        "retrieved": retrieved,
    }

    if verbose:
        print("=" * 100)
        print(f"Q: {query}")
        print(f"Expected doc_name: {expected_doc}")
        print(f"Expected evidence pages: {result['evidence_pages'] or 'N/A'}")
        print(
            f"Doc match:          {'YES' if doc_hits else 'NO'} ({len(doc_hits)}/{top_k})"
        )
        print(
            f"Evidence overlap:   {'YES' if overlap_hits else 'NO'} ({len(overlap_hits)}/{top_k})"
        )
        print(
            f"Page match:         {'YES' if page_hits else 'NO'} ({len(page_hits)}/{top_k})"
        )
        print("Top-k retrieved chunks:")
        for rank, d in enumerate(retrieved, start=1):
            print(
                f"  {rank}. {d.metadata.get('doc_name')} | page={d.metadata.get('page_number')}"
            )
            print(f"     {d.page_content[:170].replace(chr(10), ' ')}...")
        print()

    return result


# Task 3 retrieval check on 3 sample questions
retrieval_check = []

for _, row in df.head(3).iterrows():
    retrieval_check_row = check_retrieval(row, vectorstore)
    retrieval_check.append(retrieval_check_row)

retrieval_check = pd.DataFrame(retrieval_check)

Q: Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.
Expected doc_name: CORNING_2022_10K
Expected evidence pages: [59]
Doc match:          YES (3/5)
Evidence overlap:   NO (0/5)
Page match:         NO (0/5)
Top-k retrieved chunks:
  1. CORNING_2022_10K | page=101
     (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since September...
  2. CORNING_2022_10K | page=102
     (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since September...
  3. 3M_2022_10K | page=37
     not defined under U.S. generally accepted accounting principles and may not be computed the same as similarly titled measures used by other companies. The Company defines...
  4. 3M_2023

#### Task 3 — Retrieval Observations

**Right document?** Mostly yes — the 3 queries returned **3/5, 4/5, and 5/5 chunks** from the correct company's filing (12/15 overall). This makes sense because the question itself mentions the company name, and that name also appears in the chunks, so the embeddings can easily match them. The misses mostly come from cross-company chunks that talk about "working capital" in general (e.g. 3M's non-GAAP definition page) and get pulled in because that phrase dominates the query.

**Right evidence text?** No — **0/5** for all 3 questions. I opened the actual PDFs on the evidence pages and found they are mostly balance-sheet tables full of numbers. When we split these pages into 1000-char chunks, the tables get broken apart and lose their structure. The retriever pulls chunks that talk *about* the right topics, but not the exact table rows the dataset annotators marked as evidence.

**Right page?** Mostly no — only **1 out of 15** retrieved chunks came from an expected evidence page (American Water Works, page 81). The balance-sheet pages are heavy on numbers and light on regular sentences, so the embedding model doesn't "understand" them well. Instead it prefers pages with more natural language, like company overviews or notes to financial statements, that *mention* working capital in words.

**Takeaway:** The retriever generally finds the right document, but struggles to find the right *page* — especially when the answer lives in a numeric table rather than a text paragraph. This is a known limitation of dense (embedding-based) retrieval. Possible improvements: combining keyword search (BM25) with embeddings, using table-aware chunking, or first filtering by document and then searching within it.

---
## Task 4 - Building a RAG Pipeline

In [ ]:
def retrieve(query: str, k: int = 4) -> List[Dict]:
    """Embed `query` and return top-k chunks from the FAISS vector store,
    each as a flat dict with its text + key metadata (doc_name, page_number)."""
    docs = vectorstore.similarity_search(query, k=k)
    return [
        {
            "doc_name": d.metadata.get("doc_name"),
            "page_number": d.metadata.get("page_number"),
            "company": d.metadata.get("company"),
            "doc_period": d.metadata.get("doc_period"),
            "text": d.page_content,
        }
        for d in docs
    ]


# quick smoke test
sample_query = df.iloc[0]["question"]
sample_chunks = retrieve(sample_query, k=4)

print(f"Q: {sample_query}\n")

for i, c in enumerate(sample_chunks, 1):
    print(f"[{i}] {c['doc_name']} | page={c['page_number']}")
    print(f"    {c['text'][:160].replace(chr(10), ' ')}...\n")

Q: Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.

[1] CORNING_2022_10K | page=101
    (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since...

[2] CORNING_2022_10K | page=102
    (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since...

[3] 3M_2022_10K | page=37
    not defined under U.S. generally accepted accounting principles and may not be computed the same as similarly titled measures used by other companies. The Compa...

[4] 3M_2023Q2_10Q | page=70
    Current assets $ 15,754 $ 14,688 $ 1,066  Less: Current liabilities 10,936 9,523 1,413  Working capital (non-GAAP measure) $ 4,818 $ 5,165 $ (347) Various asset...



In [53]:
SYSTEM_PROMPT = """You are a careful financial-document assistant.

Rules:
- Answer ONLY using facts that appear in the provided CONTEXT below.
- If the CONTEXT does not contain the answer, reply exactly:
  "The provided context does not contain the answer."
  Do not guess and do not use outside knowledge.
- Keep answers concise (1-4 sentences).
- Cite the source document for every fact you use, in the form (doc_name, page N).
  If multiple sources support a fact, cite all of them.
"""


def build_user_prompt(query: str, chunks: List[Dict]) -> str:
    """Format retrieved chunks into a single prompt block, with clear
    separators and doc_name/page metadata so the model can cite sources.
    Handles empty retrieval explicitly."""
    if not chunks:
        context_block = "No relevant context was retrieved."
    else:
        parts = []
        for i, c in enumerate(chunks, start=1):
            header = (
                f"--- Source {i} | doc_name={c['doc_name']} "
                f"| page={c['page_number']} ---"
            )
            parts.append(f"{header}\n{c['text']}")
        context_block = "\n\n".join(parts)

    return f"CONTEXT:\n{context_block}\n\nQUESTION: {query}\n\nANSWER:"


# inspect the prompt on our sample query
print(build_user_prompt(sample_query, sample_chunks)[:1200])
print("...")

CONTEXT:
--- Source 1 | doc_name=CORNING_2022_10K | page=101 ---
(1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since September 9, 2020.  Refer to Note 3 (HSGTransactions and Acquisitions) in the notes to the consolidated financial statements for additional information.(2) Depreciation expense for Corning’s reportable segments and Hemlock and Emerging Growth Businesses includes an allocation of depreciation of corporate property not specifically identifiable to asegment.(3) Research, development and engineering expenses include direct project spending that is identifiable to a segment.(4) Income tax (provision) benefit reflects a tax rate of 21%.(5) Segment assets include inventory, accounts receivable, property, plant and equipment, net of accumulated depreciation, and associated equity companies.
 102

--- Source 2 | doc_name=CORNING_2022_10K | page=102 ---
(1) Corning obtained

In [54]:
def generate(system_prompt: str, user_prompt: str, max_tokens: int = 400) -> str:
    """Call the generation model with a system + user prompt and return
    the stripped answer string."""
    response = client.chat.completions.create(
        model=RAG_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()


# smoke test: end-to-end on the sample query, manually wiring the 3 pieces
_user_prompt = build_user_prompt(sample_query, sample_chunks)
_answer = generate(SYSTEM_PROMPT, _user_prompt)
print(f"Q: {sample_query}\n")
print(f"A: {_answer}")

Q: Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.

A: The provided context does not contain the answer.


In [ ]:
def answer_with_rag(query: str, k: int = 4) -> dict:
    """End-to-end RAG: retrieve top-k chunks from the FAISS store and
    generate a grounded, cited answer with Llama-3.3-70B-Instruct.

    Returns a dict with:
      - answer (str): the generation model's final answer.
      - retrieved_chunks (list[dict]): the chunks used as context, each with
        its doc_name, page_number, and text (plus company/doc_period for
        convenience).
    """
    retrieved_chunks = retrieve(query, k=k)
    user_prompt = build_user_prompt(query, retrieved_chunks)
    answer = generate(SYSTEM_PROMPT, user_prompt)
    return {
        "answer": answer,
        "retrieved_chunks": retrieved_chunks,
    }


# sanity test on a couple of questions from our filtered set
for _, row in df.head(2).iterrows():
    q = row["question"]
    result = answer_with_rag(q, k=4)
    print("=" * 100)
    print(f"Q: {q}")
    print(f"\nA: {result['answer']}")
    print(f"\nRetrieved {len(result['retrieved_chunks'])} chunks:")
    for i, c in enumerate(result["retrieved_chunks"], 1):
        print(f"  [{i}] {c['doc_name']} | page={c['page_number']}")
    print()

Q: Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.

A: The provided context does not contain the answer.

Retrieved 4 chunks:
  [1] CORNING_2022_10K | page=101
  [2] CORNING_2022_10K | page=102
  [3] CORNING_2022_10K | page=90
  [4] CORNING_2022_10K | page=20

Q: Does American Water Works have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.

A: To determine if American Water Works has positive working capital, we need to calculate working capital by subtracting current liabilities from current assets. However, the provided context only contains information on current liabilities (AMERICANWATERWORKS_2022_10K, page 81) and does not provide the current assets. 

The provided context does not contain the answer.

Retrieved 4 chunks:
  [1] AMERICANWATERWO

---
## Task 5 - Run and Compare

In [50]:
# Run the same 10 questions from Task 1 through the RAG pipeline,
# and save a side-by-side comparison to xlsx.

TASK_5_FILENAME = "assignment2_run_and_compare.xlsx"
TASK_5_COLUMNS = [
    "financebench_id",
    "question_type",
    "question",
    "naive_answer",
    "RAG_answer",
    "ground_truth",
]

if os.path.exists(TASK_5_FILENAME):
    print("Loading existing results...")
    compare_df = pd.read_excel(TASK_5_FILENAME)
    # rebuild the retrieved chunks separately from the cached xlsx (not persisted)
    compare_df["_rag_retrieved"] = [[] for _ in range(len(compare_df))]
else:
    rows = []
    for _, row in answers_df.iterrows():
        result = answer_with_rag(row["question"], k=4)
        rows.append(
            {
                "financebench_id": row["financebench_id"],
                "question_type": row["question_type"],
                "question": row["question"],
                "naive_answer": row["naive_answer"],
                "RAG_answer": result["answer"],
                "ground_truth": row["ground_truth"],
                "_rag_retrieved": result["retrieved_chunks"],
            }
        )
        time.sleep(1.5)  # rate limiting

    compare_df = pd.DataFrame(rows)
    # persist only the required columns
    compare_df[TASK_5_COLUMNS].to_excel(TASK_5_FILENAME, index=False)
    print(f"Saved {TASK_5_FILENAME}")

compare_df.head(1)

Loading existing results...


,financebench_id,question_type,question,naive_answer,RAG_answer,ground_truth,_rag_retrieved
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,"Based on Corning's FY2022 data, the company ha...",The provided context does not contain the answer.,Yes. Corning had a positive working capital am...,[]


In [ ]:
# Side-by-side display of naive vs RAG vs ground truth for all 10 questions.
# Also shows the retrieved (doc_name, page) pairs so we can see *why* RAG
# answered the way it did when we write the discussion.


def _fmt(text: str, width: int = 110) -> str:
    """Wrap long strings so they render nicely in notebook output."""
    import textwrap

    return "\n     ".join(textwrap.wrap(str(text), width=width)) or "(empty)"


for _, row in compare_df.iterrows():
    print("=" * 120)
    print(f"[{row['financebench_id']}] ({row['question_type']})")
    print(f"Q:   {_fmt(row['question'])}")
    print(f"\nNAIVE:   {_fmt(row['naive_answer'])}")
    print(f"\nRAG:     {_fmt(row['RAG_answer'])}")
    print(f"\nGT:      {_fmt(row['ground_truth'])}")
    retrieved = row.get("_rag_retrieved") or []

    if retrieved:
        pages = ", ".join(f"{c['doc_name']}#p{c['page_number']}" for c in retrieved)
        print(f"\nRetrieved: {pages}")

    print()

[financebench_id_00005] (domain-relevant)
Q:   Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or
     relevant metric for this company, then please state that and explain why.

NAIVE:   Based on Corning's FY2022 data, the company has positive working capital. Working capital is a relevant metric
     for Corning, as it is a manufacturing company that requires significant investments in inventory, accounts
     receivable, and accounts payable. According to Corning's FY2022 financial statements, the company's current
     assets exceed its current liabilities, indicating positive working capital. This suggests that Corning has
     sufficient liquidity to meet its short-term obligations.

RAG:     The provided context does not contain the answer.

GT:      Yes. Corning had a positive working capital amount of $831 million by FY 2022 close. This answer considers
     only operating current assets and current liabilities that were clear

### RAG vs. Naive: Analysis of the 10 Sample Comparisons

Looking at the results, here's the breakdown:

#### 1. Where RAG Helped (grounded answers over refusals/hallucinations)

| ID | Type | Why RAG won |
|---|---|---|
| `00215` Verizon capex | domain-relevant | Both right, but RAG added a verifiable citation (`VERIZON_2022_10K, page 22`) with $23.1B figure vs. naive's un-cited $22.6B |
| `00288` Best Buy cash drop | novel-generated | Naive refused outright; RAG pulled the exact numbers ($1,874 → $1,093) from `BESTBUY_2024Q2_10Q, page 19` |
| `00382` MGM EBITDAR by region | novel-generated | Both named Las Vegas, but RAG added the specific $3.142B figure and earnings-release citation |
| `00283` Pfizer/Upjohn spinoff | novel-generated | *Partial win* — naive hallucinated "$12B", RAG correctly refused. Neither produced GT ($77.78M), but RAG at least avoided a confidently wrong answer |

#### 2. Where RAG Hurt (naive memorization beat retrieval)

| ID | Type | What went wrong |
|---|---|---|
| `00005` Corning WC | domain-relevant | Naive answered "positive" (matches GT); RAG refused — retrieval likely missed Corning's balance-sheet chunk |
| `00080` PayPal WC | domain-relevant | Same pattern — naive's memorized "positive" matched GT; RAG said "context does not contain the answer" |
| `00206` JPM gross margins | domain-relevant | Naive gave a textbook-perfect explanation (gross margin irrelevant for a bank — matches GT). RAG refused. The question is *conceptual*, so retrieval couldn't help |

In all three, RAG's strict "stick to retrieved context" prompting turned a correct memorized answer into an unnecessary refusal.

#### 3. Where Both Failed

- `00070` American Water Works WC — RAG retrieved only the liabilities page (`page 81`), missing current assets; naive refused.
- `00299` JPM Q1 2021 segment revenue — both refused; specific quarterly segment data likely not in the retrieved chunks.
- `00302` Pfizer PPNE FY20→FY21 — both refused.

#### 4. Patterns by `question_type`

Tallying the 10 cases:

| Type | RAG helped | RAG hurt | Both failed | Tie |
|---|---|---|---|---|
| domain-relevant (5) | 1 (`00215`) | 3 (`00005`, `00080`, `00206`) | 1 (`00070`) | 0 |
| novel-generated (5) | 2 (`00288`, `00382`) + 1 partial (`00283`) | 0 | 2 (`00299`, `00302`) | 0 |

**RAG hurts more on `domain-relevant`, helps more on `novel-generated`.**

##### Hypothesis — why the asymmetry?

1. **`domain-relevant` questions are answerable from pretraining.** Things like "does PayPal/Corning have positive working capital?" or "are gross margins relevant for JPM?" are general-knowledge / conceptual questions the base LLM already knows. RAG's grounding prompt ("answer only from context") then *penalizes* the model: if the retrieved chunks don't literally state the figure, it refuses — even when it "knows" the answer. Retrieval thus acts as a gate that filters out correct memorized answers.

2. **`novel-generated` questions are specific, filing-level lookups** (exact cash balances, exact EBITDAR, exact segment revenue). The naive model can't memorize these — it either refuses or fabricates. RAG has a real shot because the answer literally lives in a chunk, and when retrieval works (`00288`, `00382`) the grounded citation is a big win.

3. **Retrieval recall is the bottleneck on both sides.** When RAG hurts domain-relevant questions, it's usually because the retriever surfaced *some* chunks from the right filing but not the one containing the balance-sheet figure, so the model conservatively abstains. Improving chunking (e.g. table-aware splitting) or boosting top-k for financial-statement pages would likely flip several of the "RAG hurt" cases into wins without regressing the "RAG helped" cases.

##### Summary

- RAG's gain shows up where it should: specific, filing-grounded facts with citations.
- The cost is over-conservative refusals on conceptual/common-knowledge questions. A *hybrid* policy — "answer from context if present, otherwise fall back to parametric knowledge with a clear disclaimer" — would likely dominate the current pure-RAG prompt on this mix.

---
## Task 6 - Evaluation